In [100]:
import os
from datetime import datetime, date

print(os.getcwd())

c:\Users\JiebingYin\OneDrive - Haver Analytics\python files\HolidayNoteAutomation\autohol


In [101]:
from AutoStatsFunctions import Assignments, Autocalendar

In [102]:
a = Autocalendar()
dw_sql = a.get_dw()
dw_entries = a.parse_entries()

len entries 3261


In [103]:
b = Assignments()
dw_list = b.assign(dw_entries, attach=True)

In [104]:
print(len(dw_list))
print(dw_list[:3])

3261
[{'templateID': 2257, 'database': 'INTDAILY', 'group': 'T11', 'country': 'Various', 'update': 'Asia Bonds [ABONDS]', 'time': '14:00 PM', 'auto': 'F:\\Intdaily\\Telerate\\Bonds\\asia.auto', 'trigger': 'AT', 'priority': 3, 'assign': '[AUTOT]Binru/Zukruf/Babalwa', 'procedures': 'f:\\intdaily\\telerate\\bonds\\bonds.doc', 'frq_sun': 0, 'frq_mon': 0, 'frq_tue': 0, 'frq_wed': 0, 'frq_thu': 0, 'frq_fri': 0, 'frq_sat': 0, 'frq_daily': 1, 'template': 'DW', 'usefuleness': '', 'main_team': 'HighFrequency', 'team': 'dwb', 'edm': 'binru'}, {'templateID': 2258, 'database': 'INTDAILY', 'group': 'J54, S17', 'country': 'Australia', 'update': 'Australia Stock Indexes', 'time': '02:45 AM', 'auto': 'f:\\intdaily\\aus\\asx\\asxauto.auto', 'trigger': 'AT', 'priority': 1, 'assign': '[AUTOT]Yukun/Larissa/Jesse/RyanP/Griffin', 'procedures': 'J:\\PROC\\INTL\\INTDAILY\\ANZ Capital Markets.doc', 'frq_sun': 0, 'frq_mon': 0, 'frq_tue': 0, 'frq_wed': 0, 'frq_thu': 0, 'frq_fri': 0, 'frq_sat': 0, 'frq_daily': 1, 

In [105]:
def filter_entries_for_user(entries, planner_user):
    planner_user = (planner_user or "").strip().lower()
    out = []

    for entry in entries:
        edm = (entry.get("edm") or "").strip().lower()
        country = (entry.get("country") or "").strip()

        if edm != planner_user:
            continue
        if country.lower() in ["various", "Multiple"]:
            continue

        out.append(entry)

    return out

In [106]:
user_entries = filter_entries_for_user(dw_list, "Babalwa")
print(len(user_entries))
print(user_entries[:5])

168
[{'templateID': 2264, 'database': 'INTDAILY', 'group': 'I75', 'country': 'Belgium', 'update': 'Loans in 2nd Market', 'time': '11:30 AM', 'auto': 'f:\\intdaily\\belgium\\bondy\\auto\\belgiumbondy.auto', 'trigger': 'AT', 'priority': 3, 'assign': '[AUTOT]Griffin', 'procedures': 'j:\\proc\\intl\\intdaily\\Belgium_Bond Yields.doc', 'frq_sun': 0, 'frq_mon': 0, 'frq_tue': 0, 'frq_wed': 0, 'frq_thu': 0, 'frq_fri': 0, 'frq_sat': 0, 'frq_daily': 1, 'template': 'DW', 'usefuleness': '', 'main_team': 'HighFrequency', 'team': 'dwb', 'edm': 'babalwa'}, {'templateID': 2265, 'database': 'INTDAILY', 'group': 'P85', 'country': 'Belgium', 'update': 'Gold Prices', 'time': '11:30 AM', 'auto': 'f:\\intdaily\\belgium\\gold\\auto\\belgiumgold.auto', 'trigger': 'AT', 'priority': 3, 'assign': '[AUTOT]Griffin', 'procedures': 'j:\\proc\\intl\\intdaily\\belg_gold.doc', 'frq_sun': 0, 'frq_mon': 0, 'frq_tue': 0, 'frq_wed': 0, 'frq_thu': 0, 'frq_fri': 0, 'frq_sat': 0, 'frq_daily': 1, 'template': 'DW', 'usefuleness

In [107]:
for row in user_entries[:10]:
    print(row["edm"], row["country"], row["update"])

babalwa Belgium Loans in 2nd Market
babalwa Belgium Gold Prices
babalwa Belgium Interest Rates
babalwa CANADA BoC Commodity Price Indexes
babalwa CANADA BoC Commodity Price Indexes
babalwa CANADA Bond Yields Aggregation
babalwa CANADA Montreal Exchange (Bond Futures)
babalwa CANADA S&P/TSX COMPOSITE PRICES
babalwa Euro area European Pump Prices
babalwa Euro area EONIA & EURIBOR


In [108]:
def is_template_scheduled_on_date(entry, target_date):
    """
    Return True if the template is expected to run on target_date.

    Scheduling rules:
    1. If frq_daily is 1 or -1:
       - template runs on Monday-Friday only
    2. Otherwise, use the specific weekday flags:
       - frq_sun ... frq_sat
       - a value of 1 or -1 means enabled
       - 0 means not enabled
    """

    def enabled(value) -> bool:
        return value in (1, -1)
    
    if isinstance(target_date, str):
        target_date = datetime.strptime(target_date, "%Y-%m-%d").date()

    # Rule 1: weekday schedule (Mon-Fri)
    if enabled(entry.get("frq_daily", 0)):
        return target_date.weekday() < 5   # Mon=0 ... Fri=4

    # Rule 2: specific weekday flags
    weekday_map = {
        0: "frq_mon",
        1: "frq_tue",
        2: "frq_wed",
        3: "frq_thu",
        4: "frq_fri",
        5: "frq_sat",
        6: "frq_sun",
    }

    day_field = weekday_map[target_date.weekday()]
    return enabled(entry.get(day_field, 0))


In [109]:
from datetime import date, timedelta

entry = user_entries[5]
print(entry["templateID"], entry["update"])

print(is_template_scheduled_on_date(entry, date(2026, 6, 19)))  # Friday
print(is_template_scheduled_on_date(entry, date(2026, 6, 20)))  # Saturday
print(is_template_scheduled_on_date(entry, date(2026, 6, 21)))  # Sunday

2285 Bond Yields Aggregation
False
False
False


In [110]:
def date_window(selected_date, days_before=7, days_after=7):
    return (
        selected_date - timedelta(days=days_before),
        selected_date + timedelta(days=days_after),
    )

In [127]:
from datetime import date

selected_date = date(2026, 7, 11)
window_start, window_end = date_window(selected_date)

print(selected_date)
print(window_start, window_end)

2026-07-11
2026-07-04 2026-07-18


In [112]:
def build_window_task_candidates(entries, selected_date):
    window_start, window_end = date_window(selected_date)
    rows = []

    current = window_start
    while current <= window_end:
        for entry in entries:
            if is_template_scheduled_on_date(entry, current):
                rows.append({
                    **entry,
                    "target_date": current.isoformat(),
                    "is_scheduled_on_target_date": True,
                    "is_on_holiday_date": current == selected_date,
                })
        current += timedelta(days=1)

    return rows


In [113]:
candidate_rows = build_window_task_candidates(user_entries, selected_date)
print(len(candidate_rows))
print(candidate_rows[:5])

973
[{'templateID': 2922, 'database': 'INTDAILY', 'group': 'S88', 'country': 'Germany', 'update': 'Stock Market Indicators', 'time': '00:01 AM', 'auto': 'f:\\intdaily\\germany\\stk\\auto\\germany_s88.auto', 'trigger': 'AT', 'priority': 3, 'assign': '[AUTOT]Nicholas/Himani/Ryan/GB/Griffin', 'procedures': 'j:\\proc\\intl\\intdaily\\GERMANY DAX.doc', 'frq_sun': 0, 'frq_mon': 0, 'frq_tue': -1, 'frq_wed': -1, 'frq_thu': -1, 'frq_fri': -1, 'frq_sat': -1, 'frq_daily': 0, 'template': 'DW', 'usefuleness': '', 'main_team': 'HighFrequency', 'team': 'dwb', 'edm': 'babalwa', 'target_date': '2026-07-04', 'is_scheduled_on_target_date': True, 'is_on_holiday_date': False}, {'templateID': 4720, 'database': 'INTWKLY', 'group': 'S88', 'country': 'Germany', 'update': 'DAX Volume & Value Index Agg', 'time': '00:01 AM', 'auto': 'F:\\intdaily\\germany\\dax\\W\\auto\\dax_agg.auto', 'trigger': 'NA', 'priority': 3, 'assign': '[AUTOT]AsianShift', 'procedures': 'J:\\PROC\\INTL\\intwkly\\Germany_DAX_agg.doc', 'frq_

In [114]:
for row in candidate_rows[:20]:
    print(row["templateID"], row["target_date"], row["is_on_holiday_date"], row["update"])

2922 2026-07-04 False Stock Market Indicators
4720 2026-07-04 False DAX Volume & Value Index Agg
6571 2026-07-04 False Rhine Water Level
7071 2026-07-04 False TRANSFER Rhine Water Level
7307 2026-07-04 False GLSECTOR Rhine Water Level
6070 2026-07-05 False Canola Oil Price Aggregation
6571 2026-07-05 False Rhine Water Level
7071 2026-07-05 False TRANSFER Rhine Water Level
7307 2026-07-05 False GLSECTOR Rhine Water Level
2264 2026-07-06 False Loans in 2nd Market
2265 2026-07-06 False Gold Prices
2267 2026-07-06 False Interest Rates
2290 2026-07-06 False Montreal Exchange (Bond Futures)
2293 2026-07-06 False S&P/TSX COMPOSITE PRICES
2336 2026-07-06 False EONIA & EURIBOR
2340 2026-07-06 False Debt Security and Pfandbriefe Yields
2342 2026-07-06 False Interest Rates
2481 2026-07-06 False Nominal Effective Rate
2686 2026-07-06 False BARCLAYS BOND INDEX
2759 2026-07-06 False DJSTOXX


In [115]:
import csv


def load_holiday_dict(csv_path):
    """
    Build a dict like:
      {
        "country1": [date(2026,2,16), date(2026,2,17), ...],
        "country2": [...]
      }

    Rules:
    - Reads QPP holidays CSV file
    - Converts 'Holiday Date' Excel serial -> Python date
    - Ignores rows where Holiday Observance == 'Regional' (case-insensitive exact match)
    """
    holiday_dict = {}

    with open(csv_path, "r", encoding="latin1", newline="") as f:
        reader = csv.DictReader(f, delimiter=";")

        for row in reader:
            country = (row.get("Country Name") or "").strip()
            observance = (row.get("Holiday Observance") or "").strip()

            if not country:
                continue

            # Ignore only exact "Regional" (case-insensitive)
            if observance.lower() == "regional":
                continue

            raw_date = (row.get("Holiday Date") or "").strip()
            if not raw_date:
                continue

            try:
                # CSV stores Excel serial dates (e.g., 46023)
                serial = int(float(raw_date))
                # Excel serial origin (Windows Excel)
                hdate = (datetime(1899, 12, 30) + timedelta(days=serial)).date()
            except Exception:
                continue

            key = country.lower()
            holiday_dict.setdefault(key, set()).add(hdate)

    # convert sets to sorted lists
    return {k: sorted(v) for k, v in holiday_dict.items()}


def holidays_for_country(holiday_dict, country: str):
    return holiday_dict.get(country.lower(), [])

In [116]:
csv_path = r"C:\Users\JiebingYin\OneDrive - Haver Analytics\python files\HolidayNoteAutomation\Q++ Worldwide Public Holidays ISO-2026.CSV"

holiday_dict = load_holiday_dict(csv_path)
print(len(holiday_dict))
print(list(holiday_dict.keys())[:10])

245
['abkhazia', 'afghanistan', 'åland', 'albania', 'algeria', 'american samoa', 'andorra', 'angola', 'anguilla', 'antigua and barbuda']


In [117]:
print(holiday_dict.get("canada", [])[:10])

[datetime.date(2026, 1, 1), datetime.date(2026, 1, 2), datetime.date(2026, 3, 16), datetime.date(2026, 4, 3), datetime.date(2026, 4, 5), datetime.date(2026, 4, 20), datetime.date(2026, 5, 18), datetime.date(2026, 6, 22), datetime.date(2026, 7, 1), datetime.date(2026, 7, 13)]


In [118]:
def normalize_country(country: str) -> str:
    return (country or "").strip().lower()

def is_country_holiday(holiday_dict, country, target_date):
    country_key = normalize_country(country)
    holidays = holiday_dict.get(country_key, [])
    return target_date in holidays


def annotate_holiday_context(rows, holiday_dict, selected_date):
    out = []

    for row in rows:
        row_copy = dict(row)
        row_copy["holiday_date"] = selected_date.isoformat()
        row_copy["is_country_holiday_on_selected_date"] = is_country_holiday(
            holiday_dict,
            row_copy.get("country", ""),
            selected_date
        )
        out.append(row_copy)

    return out

In [119]:
annotated_rows = annotate_holiday_context(candidate_rows, holiday_dict, selected_date)
print(annotated_rows[:20])

[{'templateID': 2922, 'database': 'INTDAILY', 'group': 'S88', 'country': 'Germany', 'update': 'Stock Market Indicators', 'time': '00:01 AM', 'auto': 'f:\\intdaily\\germany\\stk\\auto\\germany_s88.auto', 'trigger': 'AT', 'priority': 3, 'assign': '[AUTOT]Nicholas/Himani/Ryan/GB/Griffin', 'procedures': 'j:\\proc\\intl\\intdaily\\GERMANY DAX.doc', 'frq_sun': 0, 'frq_mon': 0, 'frq_tue': -1, 'frq_wed': -1, 'frq_thu': -1, 'frq_fri': -1, 'frq_sat': -1, 'frq_daily': 0, 'template': 'DW', 'usefuleness': '', 'main_team': 'HighFrequency', 'team': 'dwb', 'edm': 'babalwa', 'target_date': '2026-07-04', 'is_scheduled_on_target_date': True, 'is_on_holiday_date': False, 'holiday_date': '2026-07-11', 'is_country_holiday_on_selected_date': False}, {'templateID': 4720, 'database': 'INTWKLY', 'group': 'S88', 'country': 'Germany', 'update': 'DAX Volume & Value Index Agg', 'time': '00:01 AM', 'auto': 'F:\\intdaily\\germany\\dax\\W\\auto\\dax_agg.auto', 'trigger': 'NA', 'priority': 3, 'assign': '[AUTOT]AsianShi

In [120]:
for row in annotated_rows[:50]:
    print(
        row["country"],
        row["target_date"],
        row["holiday_date"],
        row["is_country_holiday_on_selected_date"]
    )

Germany 2026-07-04 2026-07-11 False
Germany 2026-07-04 2026-07-11 False
Germany 2026-07-04 2026-07-11 False
Germany 2026-07-04 2026-07-11 False
Germany 2026-07-04 2026-07-11 False
CANADA 2026-07-05 2026-07-11 False
Germany 2026-07-05 2026-07-11 False
Germany 2026-07-05 2026-07-11 False
Germany 2026-07-05 2026-07-11 False
Belgium 2026-07-06 2026-07-11 True
Belgium 2026-07-06 2026-07-11 True
Belgium 2026-07-06 2026-07-11 True
CANADA 2026-07-06 2026-07-11 False
CANADA 2026-07-06 2026-07-11 False
Euro area 2026-07-06 2026-07-11 False
Germany 2026-07-06 2026-07-11 False
Germany 2026-07-06 2026-07-11 False
Euro area 2026-07-06 2026-07-11 False
US 2026-07-06 2026-07-11 False
US 2026-07-06 2026-07-11 False
Euro area 2026-07-06 2026-07-11 False
Luxembourg 2026-07-06 2026-07-11 False
Euro area 2026-07-06 2026-07-11 False
Malta 2026-07-06 2026-07-11 False
Euro area 2026-07-06 2026-07-11 False
Euro area 2026-07-06 2026-07-11 False
CANADA 2026-07-06 2026-07-11 False
CANADA 2026-07-06 2026-07-11 Fal

In [121]:
def build_planner_rows(candidate_rows, planner_user, selected_date):
    window_start, window_end = date_window(selected_date)
    created_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    planner_rows = []
    for row in candidate_rows:
        planner_rows.append({
            "planner_user": planner_user,
            "plan_created_at": created_at,
            "selected_date": selected_date.isoformat(),
            "window_start": window_start.isoformat(),
            "window_end": window_end.isoformat(),

            "templateID": row.get("templateID", ""),
            "database": row.get("database", ""),
            "country": row.get("country", ""),
            "group": row.get("group", ""),
            "update": row.get("update", ""),
            "time": row.get("time", ""),
            "edm": row.get("edm", ""),

            "holiday_date": row.get("holiday_date", ""),
            "target_date": row.get("target_date", ""),

            "planned_action": "",
            "planned_note": "",
            "move_mode": "",
            "move_to_date": "",
            "plan_status": "PLANNED",
        })

    return planner_rows


In [122]:
planner_rows = build_planner_rows(annotated_rows, "Babalwa", selected_date)
print(len(planner_rows))
print(planner_rows[:3])

973
[{'planner_user': 'Babalwa', 'plan_created_at': '2026-03-22 16:40:51', 'selected_date': '2026-07-11', 'window_start': '2026-07-04', 'window_end': '2026-07-18', 'templateID': 2922, 'database': 'INTDAILY', 'country': 'Germany', 'group': 'S88', 'update': 'Stock Market Indicators', 'time': '00:01 AM', 'edm': 'babalwa', 'holiday_date': '2026-07-11', 'target_date': '2026-07-04', 'planned_action': '', 'planned_note': '', 'move_mode': '', 'move_to_date': '', 'plan_status': 'PLANNED'}, {'planner_user': 'Babalwa', 'plan_created_at': '2026-03-22 16:40:51', 'selected_date': '2026-07-11', 'window_start': '2026-07-04', 'window_end': '2026-07-18', 'templateID': 4720, 'database': 'INTWKLY', 'country': 'Germany', 'group': 'S88', 'update': 'DAX Volume & Value Index Agg', 'time': '00:01 AM', 'edm': 'babalwa', 'holiday_date': '2026-07-11', 'target_date': '2026-07-04', 'planned_action': '', 'planned_note': '', 'move_mode': '', 'move_to_date': '', 'plan_status': 'PLANNED'}, {'planner_user': 'Babalwa', '

In [125]:
import json


def write_planner_log_json(path, rows):
    folder = os.path.dirname(path)
    if folder:
        os.makedirs(folder, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

    return len(rows)

In [ ]:
json_count = write_planner_log_json(
    r"C:\Users\JiebingYin\OneDrive - Haver Analytics\python files\HolidayNoteAutomation\data\logs\planner_log.json", planner_rows)


In [129]:
def write_planner_log_csv(path, rows):
    folder = os.path.dirname(path)
    if folder:
        os.makedirs(folder, exist_ok=True)

    fieldnames = [
        "planner_user",
        "plan_created_at",
        "selected_date",
        "window_start",
        "window_end",
        "templateID",
        "database",
        "country",
        "group",
        "update",
        "time",
        "edm",
        "holiday_date",
        "target_date",
        "planned_action",
        "planned_note",
        "move_mode",
        "move_to_date",
        "plan_status",
    ]

    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    return len(rows)

In [131]:
csv_count = write_planner_log_csv(
    r"C:\Users\JiebingYin\OneDrive - Haver Analytics\python files\HolidayNoteAutomation\data\logs\planner_log.csv", planner_rows)